# GPU worker for photoreal splats

This notebook turns a free Colab GPU into a **worker** for your platform. It connects to the
worker gateway on your laptop, picks up every project that asked for a *photoreal Gaussian
splat*, trains it with Nerfstudio **splatfacto**, and uploads the result. The splat then
appears in the 3D Studio's **Walk** mode, with no downloading or copying by hand.

```text
Your laptop                                   Colab (this notebook)
  web platform  :8000                          ┌───────────────────────────┐
  worker gateway :8001 ◄── cloudflared tunnel ─┤ python -m queue.workers   │
  mesh worker                                  │   --pipeline splat        │
                                               │ claim → download video →  │
                                               │ COLMAP → splatfacto →     │
                                               │ upload point_cloud.ply    │
                                               └───────────────────────────┘
```

**Before you start, on your laptop:** the API, the worker gateway, and the tunnel are running
(README → *Photoreal splats on a GPU*). You need the tunnel URL and your `WORKER_TOKEN`.
Keep the laptop **awake, lid open, and plugged in** for the whole run; if it sleeps, the tunnel
dies. On macOS run `caffeinate -dims` in a spare terminal.

**Runtime → Change runtime type → T4 GPU** (or better), then run the cells in order.

## 1. Check the GPU

In [ ]:
import subprocess, torch
assert torch.cuda.is_available(), "Runtime → Change runtime type → GPU, then run this cell again"
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

## 2. Install COLMAP, Nerfstudio, and this project (about 5–10 minutes)

In [ ]:
!apt-get -qq update && apt-get -qq install -y colmap ffmpeg > /dev/null
!pip -q install nerfstudio
import os
REPO = "/content/ai-indoor-3d-reconstruction"
if not os.path.isdir(REPO):
    !git clone -q https://github.com/prakrity26/ai-indoor-3d-reconstruction.git {REPO}
else:
    !git -C {REPO} pull -q
# --no-deps keeps Nerfstudio's pinned numpy/torch versions; the worker only needs httpx + sqlalchemy.
!pip -q install --no-deps -e {REPO}
!pip -q install httpx "sqlalchemy>=2"
os.chdir(REPO)
!which ns-train ns-process-data ns-export colmap ffmpeg

## 3. Connect to your laptop

Paste the tunnel URL (looks like `https://something.trycloudflare.com`) and the worker token.
The token is read with `getpass`, so it is not saved in the notebook. If you stored it in
Colab **Secrets** (key icon on the left) as `WORKER_TOKEN`, it is used automatically.

**Keep `KEEP_IN_DRIVE = True`.** The worker then saves the camera poses and training checkpoints
to `MyDrive/indoor_splat/<project>/resume/`. If Colab disconnects, nothing is lost: reconnect,
rerun cells 1–4, press **Train again** on the project page, and the worker skips COLMAP and
continues training from the last checkpoint (saved every 2,000 steps). Each finished splat is
also copied to `MyDrive/indoor_splat/<project>/point_cloud.ply`.

In [ ]:
import os
from getpass import getpass

os.environ["WORKER_GATEWAY_URL"] = input("Tunnel URL (https://…trycloudflare.com): ").strip().rstrip("/")
try:
    from google.colab import userdata
    os.environ["WORKER_TOKEN"] = userdata.get("WORKER_TOKEN")
    print("Using WORKER_TOKEN from Colab Secrets")
except Exception:
    os.environ["WORKER_TOKEN"] = getpass("WORKER_TOKEN: ").strip()

KEEP_IN_DRIVE = True
if KEEP_IN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["SPLAT_KEEP_ROOT"] = "/content/drive/MyDrive/indoor_splat"

import httpx
r = httpx.get(os.environ["WORKER_GATEWAY_URL"] + "/worker/ping",
              headers={"Authorization": "Bearer " + os.environ["WORKER_TOKEN"]}, timeout=30)
print("gateway:", r.status_code, r.text)
assert r.status_code == 200, "Check the tunnel URL and that WORKER_TOKEN matches the laptop."

## 3b. Build the GPU kernels (first session ~10–20 min, then cached)

Splatfacto's rasterizer (gsplat) compiles CUDA code the first time it is used. Doing it here,
two compile jobs at a time, avoids the out-of-memory crash (`Killed`, exit 137) that happens
when it compiles everything at once in the middle of a job. With Drive enabled the result is
cached in `MyDrive/indoor_splat/_torch_extensions`, so later sessions take seconds.

In [ ]:
import os, shutil, subprocess, sys, time

os.environ["MAX_JOBS"] = "2"
local_cache = os.path.expanduser("~/.cache/torch_extensions")
drive_cache = "/content/drive/MyDrive/indoor_splat/_torch_extensions"
use_drive = bool(os.environ.get("SPLAT_KEEP_ROOT"))

if use_drive and os.path.isdir(drive_cache) and not os.path.isdir(local_cache):
    shutil.copytree(drive_cache, local_cache)
    print("restored cached kernels from Drive")

start = time.time()
print("Building gsplat CUDA kernels (skipped if already built)…", flush=True)
# cwd=/content: inside the repo its queue/ package would shadow the stdlib module torch needs.
build = subprocess.run([sys.executable, "-c", "from gsplat.cuda._backend import _C; print('gsplat kernels ready')"],
                       cwd="/content", capture_output=True, text=True)
print(build.stdout[-2000:], build.stderr[-4000:])
assert build.returncode == 0, "Kernel build failed; see the error above."
print(f"done in {(time.time() - start) / 60:.1f} min")

if use_drive and os.path.isdir(local_cache):
    shutil.copytree(local_cache, drive_cache, dirs_exist_ok=True)
    print("kernels cached on Drive for next time")

## 4. Run the worker

Leave this cell running. It trains queued splats one after another and waits for new ones.
Progress shows live on the laptop (Processing jobs and the project page). Stop it with the
■ button; the current job is then marked failed and can be trained again.

Settings: `SPLAT_NUM_FRAMES` (frames sampled from the video, default 180),
`SPLAT_MAX_SIDE` (frames are shrunk to this long side before COLMAP, default 1600, which keeps
COLMAP to minutes on Colab's CPU), and `SPLAT_TRAIN_STEPS` (default 30000; 7000 gives a quick
preview in about a quarter of the time).

In [ ]:
import os
os.environ.setdefault("SPLAT_NUM_FRAMES", "180")
os.environ.setdefault("SPLAT_TRAIN_STEPS", "30000")
os.environ.setdefault("SPLAT_MAX_SIDE", "1600")
os.environ["SPLAT_WORK_DIR"] = "/content/splat_work"
gpu = __import__("torch").cuda.get_device_name(0).replace(" ", "-")
!python -m queue.workers --pipeline splat --backend splatfacto --name colab-{gpu}

## Troubleshooting

| Symptom | Fix |
|---|---|
| `gateway: 401` | The token differs from `WORKER_TOKEN` on the laptop. |
| `gateway` connection error | The tunnel or `python -m api.worker_gateway` stopped on the laptop; restart it and paste the **new** URL (quick tunnels change URL every start). |
| *Splat failed: … only N MB* | Too few frames registered. Film slower, with more overlap, and train again. |
| *GPU worker stopped responding* | Colab disconnected. Reconnect, rerun cells 1–4, and press **Train again** on the project page. With `KEEP_IN_DRIVE = True` it resumes: the log prints `resumed: camera poses …` and `resumed: training from …`. |
| Log shows *Estimating camera poses* for a long time with no output | Normal: COLMAP prints nothing until it finishes. The `progress … 200 OK` lines every minute show the worker is alive. |
| Out of GPU memory | Lower `SPLAT_NUM_FRAMES` (e.g. 120) and rerun cell 4. |
| `Killed` / exit 137 while building `gsplat_cuda` | The CUDA compile ran out of RAM. Run cell 3b (it limits the build to 2 jobs) before cell 4. |
| `530` or `Name or service not known` in the log | The laptop went to sleep or the tunnel stopped. The worker keeps retrying; keep the laptop awake (lid open, plugged in, `caffeinate`). If you restarted `cloudflared`, its URL changed: stop cell 4, rerun cells 3 and 4 with the new URL. |